In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\bradl\Documents\Projets Perso\real-estate-ai-agent


In [2]:
import polars as pl
from app.core.paths import BRONZE_DIR
import requests
import time
import re
import unicodedata

***


# DVF bronze cleaning


### 1. Import

In [25]:
dvf_df = pl.read_csv(BRONZE_DIR / "dvf.csv",
                     separator=",",
                     infer_schema_length=0
                     )

In [26]:
dvf_df.shape

(395612, 41)

In [22]:
dvf_df.head()

"id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe"
str
"""2020-815165,2020-10-08,1,Vente…"
"""2020-814507,2020-07-02,1,Vente…"
"""2020-814508,2020-07-01,1,Vente…"
"""2020-814509,2020-07-01,1,Vente…"
"""2020-814509,2020-07-01,1,Vente…"


In [13]:
# Remove duplicate header rows
dvf_df = dvf_df.remove(dvf_df["date_mutation"].str.contains("[a-zA-Z]"))

### 2. Typage des colonnes

In [14]:
DATE_COLS = ["date_mutation"]

In [15]:
INT_COLS = [
    "numero_disposition",
    "numero_volume", 
    "nombre_lots",
    "nombre_pieces_principales"
]

In [16]:
FLOAT_COLS = [
    "valeur_fonciere",
    "lot1_surface_carrez",
    "lot2_surface_carrez",
    "lot3_surface_carrez",
    "lot4_surface_carrez",
    "lot5_surface_carrez",
    "surface_reelle_bati",
    "surface_terrain",
    "longitude",
    "latitude"
]

In [17]:
STR_COLS = [
    "id_mutation",
    "nature_mutation",
    "adresse_numero",
    "adresse_suffixe",
    "adresse_nom_voie",
    "adresse_code_voie",
    "code_postal",
    "code_commune",
    "nom_commune",
    "code_departement",
    "ancien_code_commune",
    "id_parcelle",
    "ancien_id_parcelle",
    "lot1_numero", 
    "lot2_numero", 
    "lot3_numero",
    "lot4_numero",
    "lot5_numero",
    "code_type_local",
    "lot5_surface_carrez",
    "type_local",
    "code_nature_culture",	
    "nature_culture",
    "code_nature_culture_speciale",
    "nature_culture_speciale",
    "section_prefixe"
]

In [ ]:
dvf_silver = dvf_df.with_columns(
    pl.col(DATE_COLS).str.to_date("%Y-%m-%d"),
    pl.col(INT_COLS).cast(pl.Int64),
    pl.col(FLOAT_COLS)
        .str.replace_all(",", ".")
        .cast(pl.Float64),
)

In [19]:
dvf_silver.sample(5)

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe
str,date,i64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,f64,str,f64,str,f64,str,f64,str,f64,i64,str,str,f64,i64,str,str,str,str,f64,f64,f64,str
"""2023-1354546""",2023-09-22,1,"""Vente""",7.6e6,"""6""",null,"""RUE FERRUS""","""3623""","""75014""","""75114""","""Paris 14e Arrondissement""","""75""",null,null,"""75114000AT0048""",null,null,"""192""",null,null,null,null,null,null,null,null,null,1,"""3""","""Dépendance""",null,0,null,null,null,null,null,2.339769,48.830995,"""000AT"""
"""2020-829283""",2020-12-16,1,"""Vente""",475000.0,"""34""",null,"""RUE TRUFFAUT""","""9487""","""75017""","""75117""","""Paris 17e Arrondissement""","""75""",null,null,"""75117000CO0031""",null,null,"""29""",44.3,null,null,null,null,null,null,null,null,1,"""2""","""Appartement""",42.0,2,null,null,null,null,null,2.322154,48.885807,"""000CO"""
"""2022-1662934""",2022-12-23,1,"""Vente""",300000.0,"""52""",null,"""RUE JOSEPH DE MAISTRE""","""5010""","""75018""","""75118""","""Paris 18e Arrondissement""","""75""",null,null,"""75118000AK0110""",null,null,"""26""",25.7,"""28""",null,null,null,null,null,null,null,2,"""2""","""Appartement""",26.0,1,null,null,null,null,null,2.331786,48.889521,"""000AK"""
"""2024-1201665""",2024-09-05,1,"""Vente""",1.16e6,"""33""",null,"""RUE RENNEQUIN""","""8149""","""75017""","""75117""","""Paris 17e Arrondissement""","""75""",null,null,"""75117000AM0013""",null,null,"""1054""",null,null,null,null,null,null,null,null,null,1,"""3""","""Dépendance""",null,0,null,null,null,null,null,2.296977,48.881869,"""000AM"""
"""2025-500758""",2025-03-31,1,"""Vente""",9.75e7,"""83""",null,"""RUE DE GRENELLE""","""4314""","""75007""","""75107""","""Paris 7e Arrondissement""","""75""",null,null,"""75107000AL0017""",null,null,null,null,null,null,null,null,null,null,null,null,0,"""3""","""Dépendance""",null,0,"""AG""","""terrains d'agrément""","""PARC""","""Parc""",1250.0,2.322118,48.855697,"""000AL"""


In [13]:
dvf_silver.shape

(395605, 41)

### 3. Valeurs uniques colonnes textuelles

In [14]:
text_cols = [
    "nature_mutation",
    "adresse_nom_voie",
    "code_postal",
    "nom_commune",
    "code_departement",
    "type_local"
    ]

In [15]:
# Occurences des observations pour certaines colonnes textuels
for i in text_cols:
    print(f"Fréquence des catégories pour la colonne: {i}")
    print(dvf_silver[i].value_counts(sort=True))
    print("-----------------------------------------------\n")

Fréquence des catégories pour la colonne: nature_mutation
shape: (6, 2)
┌─────────────────────────────────┬────────┐
│ nature_mutation                 ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Vente                           ┆ 388302 │
│ Vente en l'état futur d'achève… ┆ 3290   │
│ Echange                         ┆ 3127   │
│ Adjudication                    ┆ 837    │
│ Vente terrain à bâtir           ┆ 46     │
│ Expropriation                   ┆ 3      │
└─────────────────────────────────┴────────┘
-----------------------------------------------

Fréquence des catégories pour la colonne: adresse_nom_voie
shape: (3_872, 2)
┌───────────────────────┬───────┐
│ adresse_nom_voie      ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ RUE LECOURBE          ┆ 2531  │
│ RUE DE VAUGIRARD      ┆ 2260  │
│ RUE SAINT MAUR        

### 4. Traitement des données manquantes

In [16]:
dvf_silver.null_count()

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,5230,1394,378390,1327,1327,1336,0,0,0,395605,395605,0,395605,394157,40253,241150,233317,350176,368575,391046,385728,394450,391442,395187,0,4207,4207,180610,4271,356808,356808,395209,395209,356808,580,580,0


In [17]:
crit_cols = [
    "valeur_fonciere",
    "adresse_numero",
    "adresse_nom_voie",
    "code_postal",
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales"
    ]

- #### code_postal

In [66]:
# Utilisation de code commune (aucun null) pour déduire code postal
dvf_silver = dvf_silver.with_columns(
    pl.when(
        pl.col("code_postal").is_null()
        & pl.col("code_commune").is_not_null()
    )
    .then(
        pl.lit("750") + pl.col("code_commune").str.slice(-2)
    )
    .otherwise(pl.col("code_postal"))
    .alias("code_postal")
)

In [19]:
dvf_silver.shape

(395605, 41)

- #### adresse_nom_voie

In [67]:
# Adresse manquantes se présentant comme des doublons ou plus
adrs_srch = dvf_silver.group_by(["id_mutation","code_postal","valeur_fonciere"]).agg(pl.col("adresse_nom_voie").unique())
sus_adrs = adrs_srch.filter(pl.col("adresse_nom_voie").list.contains(None) &
                 (pl.col("adresse_nom_voie").list.len() >= 2))

In [68]:
# id de mutation des lignes se présentant comme des doublons
ids = sus_adrs.get_column("id_mutation").unique().to_list()

In [69]:
# Ajout d'une colonne flag pour les lignes suspectes
dvf_silver = dvf_silver.with_columns(
    pl.when(pl.col("id_mutation").is_in(ids)
            & pl.col("adresse_nom_voie").is_null()
            )
    .then(True)
    .otherwise(False)
    .alias("suspect_rows")
)

In [70]:
# Drop des lignes suspectes
dvf_silver = dvf_silver.filter(~pl.col("suspect_rows"))

In [24]:
dvf_silver.shape

(395463, 42)

- #### adresse_numero

In [71]:
# Adresse manquantes se présentant comme des doublons ou plus
adrs_srch = dvf_silver.group_by(["id_mutation","valeur_fonciere","adresse_nom_voie"]).agg(pl.col("adresse_numero").unique())
sus_adrs = adrs_srch.filter(pl.col("adresse_numero").list.contains(None)
                            & (pl.col("adresse_numero").list.len() >= 2))

In [72]:
# id de mutation des lignes se présentant comme des doublons
ids = sus_adrs.get_column("id_mutation").unique().to_list()

In [73]:
# Ajout d'une colonne flag pour les lignes suspectes
dvf_silver = dvf_silver.with_columns(
    pl.when(pl.col("id_mutation").is_in(ids)
            & pl.col("adresse_numero").is_null()
            )
    .then(True)
    .otherwise(False)
    .alias("suspect_rows")
)

In [74]:
# Drop des lignes suspectes
dvf_silver = dvf_silver.filter(~pl.col("suspect_rows"))

In [29]:
dvf_silver.shape

(395456, 42)

In [ ]:
rows_to_impute = (
    dvf_silver
    .filter((
            pl.col("adresse_numero").is_null()
            | pl.col("adresse_nom_voie").is_null()) 
            & pl.col("longitude").is_not_null() 
            & pl.col("latitude").is_not_null()
            )
    .select(["id_mutation", "longitude", "latitude", "code_postal", "code_commune"])
)

In [31]:
BASE_URL = "https://data.geopf.fr/geocodage/reverse"

def reverse_geocode_fr(lon, lat, postcode=None, citycode=None):
    params = {
        "index": "address",
        "lon": lon,
        "lat": lat,
        "limit": 1,
    }

    if postcode is not None:
        params["postcode"] = str(postcode)
    if citycode is not None:
        params["citycode"] = str(citycode)

    r = requests.get(BASE_URL, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()

    features = data.get("features", [])
    if not features:
        return None

    props = features[0]["properties"]

    return {
        "full_address_geocoded": props.get("label"),
        "street_geocoded": props.get("street"),
        "housenumber_geocoded": props.get("housenumber"),
        "postcode_geocoded": props.get("postcode"),
        "city_geocoded": props.get("city"),
        "citycode_geocoded": props.get("citycode"),
        "score_geocoded": props.get("score"),
    }

In [32]:
results = []

for row in rows_to_impute.iter_rows(named=True):
    try:
        geo = reverse_geocode_fr(
            lon=row["longitude"],
            lat=row["latitude"],
            postcode=row["code_postal"],
        )

        if geo is None:
            results.append({
                "id_mutation": row["id_mutation"],
                "street_geocoded": None,
                "postcode_geocoded": None,
                "citycode_geocoded": None,
                "score_geocoded": None,
                "address_inferred_from_coords": False,
                "lon": row["longitude"],
                "lat": row["latitude"]
            })
            continue

        results.append({
            "id_mutation": row["id_mutation"],
            **geo,
            "address_inferred_from_coords": True,
            "lon": row["longitude"],
            "lat": row["latitude"]
        })

        time.sleep(0.05)  # petite pause prudente
    except Exception:
        results.append({
            "id_mutation": row["id_mutation"],
            "street_geocoded": None,
            "postcode_geocoded": None,
            "citycode_geocoded": None,
            "score_geocoded": None,
            "address_inferred_from_coords": False,
            "lon": row["longitude"],
            "lat": row["latitude"]
        })

In [75]:
geo_df = pl.DataFrame(results)

In [34]:
geo_df.head()

id_mutation,full_address_geocoded,street_geocoded,housenumber_geocoded,postcode_geocoded,city_geocoded,citycode_geocoded,score_geocoded,address_inferred_from_coords,lon,lat
str,str,str,str,str,str,str,f64,bool,f64,f64
"""2020-815165""","""140a Rue de Rivoli 75001 Paris""","""Rue de Rivoli""","""140a""","""75001""","""Paris""","""75101""",0.9983,true,2.343056,48.86059
"""2020-815165""","""140a Rue de Rivoli 75001 Paris""","""Rue de Rivoli""","""140a""","""75001""","""Paris""","""75101""",0.9983,true,2.343056,48.86059
"""2020-815165""","""140a Rue de Rivoli 75001 Paris""","""Rue de Rivoli""","""140a""","""75001""","""Paris""","""75101""",0.9983,true,2.343056,48.86059
"""2020-815165""","""140a Rue de Rivoli 75001 Paris""","""Rue de Rivoli""","""140a""","""75001""","""Paris""","""75101""",0.9983,true,2.343056,48.86059
"""2020-815126""","""140a Rue de Rivoli 75001 Paris""","""Rue de Rivoli""","""140a""","""75001""","""Paris""","""75101""",0.9983,true,2.343056,48.86059


In [35]:
geo_df.filter(pl.col("housenumber_geocoded").str.contains("[a-zA-Z]")).sample(5)

id_mutation,full_address_geocoded,street_geocoded,housenumber_geocoded,postcode_geocoded,city_geocoded,citycode_geocoded,score_geocoded,address_inferred_from_coords,lon,lat
str,str,str,str,str,str,str,f64,bool,f64,f64
"""2020-822649""","""1p Rue Alphonse Boudard 75013 …","""Rue Alphonse Boudard""","""1p""","""75013""","""Paris""","""75113""",0.9989,true,2.373495,48.832038
"""2021-1702892""","""163p1 Avenue de France 75013 P…","""Avenue de France""","""163p1""","""75013""","""Paris""","""75113""",0.9977,true,2.373369,48.833202
"""2022-1674504""","""46p Rue Blomet 75015 Paris""","""Rue Blomet""","""46p""","""75015""","""Paris""","""75115""",0.9987,true,2.305577,48.842798
"""2023-1352006""","""31a Rue Bobillot 75013 Paris""","""Rue Bobillot""","""31a""","""75013""","""Paris""","""75113""",0.9947,true,2.354633,48.829151
"""2024-1198208""","""33p Rue Piat 75020 Paris""","""Rue Piat""","""33p""","""75020""","""Paris""","""75120""",0.9981,true,2.383828,48.872324


In [76]:
geo_df = geo_df.with_columns(
    pl.when(pl.col("housenumber_geocoded").str.contains("[a-zA-Z]"))
    .then(
        pl.col("housenumber_geocoded").str.extract(r"^(\d+)", group_index=1)
    )
    .otherwise(pl.col("housenumber_geocoded"))
    .alias("adresse_numero_clean")
)

In [77]:
ABBREVIATIONS = {
    "BOULEVARD": "BD",
    "PASSAGE": "PAS",
    "AVENUE": "AV",
    "IMPASSE": "IMP",
    "PLACE": "PL",
    "ROUTE": "RTE",
    "ALLEE": "ALL",
    "SQUARE": "SQ",
    "VILLA": "VLA",
    "FAUBOURG": "FG",
    "CHEMIN": "CHE",
}

In [38]:
def normalize_voie(s: str) -> str:
    if s is None:
        return None

    # Uppercase
    s = s.upper()

    # Suppression des accents
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if not unicodedata.combining(c)
    )

    # Apostrophes / tirets / séparateurs -> espace
    s = re.sub(r"['’`´\-_/.,;:()]", " ", s)

    # Abréviations
    for mot, abbr in ABBREVIATIONS.items():
        s = re.sub(rf"\b{mot}\b", abbr, s)

    # Ne garder que lettres, chiffres et espaces
    s = re.sub(r"[^A-Z0-9 ]", " ", s)

    # Normalisation des espaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [39]:
geo_df.shape

(1242, 12)

In [78]:
col_to_keep = [
    "id_mutation",
    "postcode_geocoded",
    "score_geocoded",
    "address_inferred_from_coords",
    "lon",
    "lat",
    "adresse_numero_clean",
    "street_geocoded"
    ]

In [79]:
geo_df = geo_df.unique(subset=["id_mutation","lon","lat"], maintain_order=True)

In [80]:
dvf_silver = dvf_silver.join(geo_df[col_to_keep],
                                   left_on=["id_mutation","longitude","latitude"],
                                   right_on=["id_mutation","lon","lat"],
                                   how="left",
                                   validate="m:1"
                                   )

In [81]:
dvf_silver = dvf_silver.with_columns(
    pl.when(pl.col("address_inferred_from_coords").is_null())
    .then(False)
    .otherwise(True)
    .alias("address_inferred_from_coords")
    ,
    pl.when(pl.col("address_inferred_from_coords") & 
            (pl.col("score_geocoded") > 0.9))
    .then(pl.col("adresse_numero_clean"))
    .otherwise(pl.col("adresse_numero"))
    .alias("adresse_numero")
    ,
    pl.when(pl.col("address_inferred_from_coords") &
            (pl.col("score_geocoded") > 0.9))
    .then(pl.col("street_geocoded"))
    .otherwise(pl.col("adresse_nom_voie"))
    .alias("adresse_nom_voie")
)

In [82]:
dvf_silver = dvf_silver.with_columns(
    pl.col("adresse_nom_voie")
    .map_elements(normalize_voie, return_dtype=pl.String)
    .alias("adresse_nom_voie")
)

In [45]:
dvf_silver.null_count()

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe,suspect_rows,postcode_geocoded,score_geocoded,address_inferred_from_coords,adresse_numero_clean,street_geocoded
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,5229,27,378241,27,1185,0,0,0,0,395456,395456,0,395456,394141,40115,241001,233168,350027,368426,390897,385579,394301,391293,395038,0,4058,4058,180461,4122,356664,356664,395060,395060,356664,579,579,0,0,394208,394208,0,394232,394232


In [46]:
geo_df.null_count()

id_mutation,full_address_geocoded,street_geocoded,housenumber_geocoded,postcode_geocoded,city_geocoded,citycode_geocoded,score_geocoded,address_inferred_from_coords,lon,lat,adresse_numero_clean
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,11,11,0,0,0,0,0,0,0,11


- #### valeur_fonciere, surface_relle_bati, nombre_pieces_principales

In [49]:
crit_cols

['valeur_fonciere',
 'adresse_numero',
 'adresse_nom_voie',
 'code_postal',
 'type_local',
 'surface_reelle_bati',
 'nombre_pieces_principales']

In [50]:
dvf_silver.null_count()

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe,suspect_rows,postcode_geocoded,score_geocoded,address_inferred_from_coords,adresse_numero_clean,street_geocoded
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,5229,0,378241,0,1185,0,0,0,0,395456,395456,0,395456,394141,40115,241001,233168,350027,368426,390897,385579,394301,391293,395038,0,4058,4058,180461,4122,356664,356664,395060,395060,356664,579,579,0,0,394208,394208,0,394232,394232


In [84]:
dvf_silver = dvf_silver.with_columns(
    pl.col("valeur_fonciere").is_null().alias("is_valeur_fonciere_missing"),
    pl.col("surface_reelle_bati").is_null().alias("is_surface_missing"),
    pl.col("nombre_pieces_principales").is_null().alias("is_nombre_pieces_missing")
)

- #### type_local

In [85]:
dvf_silver = dvf_silver.with_columns(
    pl.when(pl.col("type_local").is_null())
    .then(pl.lit("Indisponible"))
    .otherwise(pl.col("type_local"))
    .alias("type_local")
)

In [53]:
dvf_silver

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,ancien_code_commune,ancien_nom_commune,id_parcelle,ancien_id_parcelle,numero_volume,lot1_numero,lot1_surface_carrez,lot2_numero,lot2_surface_carrez,lot3_numero,lot3_surface_carrez,lot4_numero,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe,suspect_rows,postcode_geocoded,score_geocoded,address_inferred_from_coords,adresse_numero_clean,street_geocoded,vf_missing_flag,sr_missing_flag,np_missing_flag
str,date,i64,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,f64,str,f64,str,f64,str,f64,str,f64,i64,str,str,f64,i64,str,str,str,str,f64,f64,f64,str,bool,str,f64,bool,str,str,bool,bool,bool
"""2020-815165""",2020-10-08,1,"""Vente""",1.26e8,"""140""",null,"""RUE DE RIVOLI""",null,"""75001""","""75101""","""Paris 1er Arrondissement""","""75""",null,null,"""75101000AT0104""",null,23,null,null,null,null,null,null,null,null,null,null,0,null,"""Indisponible""",null,null,null,null,null,null,null,2.343056,48.86059,"""000AT""",false,"""75001""",0.9983,true,"""140""","""Rue de Rivoli""",false,true,true
"""2020-814507""",2020-07-02,1,"""Vente""",45000.0,"""9""",null,"""RUE DES ARQUEBUSIERS""","""0465""","""75003""","""75103""","""Paris 3e Arrondissement""","""75""",null,null,"""75103000AM0116""",null,null,"""3650""",null,null,null,null,null,null,null,null,null,1,"""3""","""Dépendance""",null,0,null,null,null,null,null,2.365997,48.859077,"""000AM""",false,null,null,false,null,null,false,true,false
"""2020-814508""",2020-07-01,1,"""Vente""",148000.0,"""105""",null,"""RUE VIEILLE DU TEMPLE""","""9783""","""75003""","""75103""","""Paris 3e Arrondissement""","""75""",null,null,"""75103000AQ0062""",null,null,"""4""",11.81,null,null,null,null,null,null,null,null,1,"""2""","""Appartement""",12.0,1,null,null,null,null,null,2.361559,48.860524,"""000AQ""",false,null,null,false,null,null,false,false,false
"""2020-814509""",2020-07-01,1,"""Vente""",90000.0,"""47""",null,"""RUE DE COURCELLES""","""2387""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",null,null,"""75108000BC0018""",null,null,"""30""",5.7,null,null,null,null,null,null,null,null,1,"""3""","""Dépendance""",null,0,null,null,null,null,null,2.307423,48.876371,"""000BC""",false,null,null,false,null,null,false,true,false
"""2020-814509""",2020-07-01,1,"""Vente""",90000.0,"""47""",null,"""RUE DE COURCELLES""","""2387""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",null,null,"""75108000BC0018""",null,null,"""31""",11.03,null,null,null,null,null,null,null,null,1,"""3""","""Dépendance""",null,0,null,null,null,null,null,2.307423,48.876371,"""000BC""",false,null,null,false,null,null,false,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2024-1213647""",2024-04-22,1,"""Vente""",1.5e6,"""29""",null,"""RUE LHOMOND""","""5648""","""75005""","""75105""","""Paris 5e Arrondissement""","""75""",null,null,"""75105000AZ0084""",null,null,"""18""",null,null,null,null,null,null,null,null,null,1,"""2""","""Appartement""",15.0,1,null,null,null,null,null,2.347559,48.843111,"""000AZ""",false,null,null,false,null,null,false,false,false
"""2024-1213647""",2024-04-22,1,"""Vente""",1.5e6,"""29""",null,"""RUE LHOMOND""","""5648""","""75005""","""75105""","""Paris 5e Arrondissement""","""75""",null,null,"""75105000AZ0084""",null,null,"""19""",null,"""21""",null,null,null,null,null,null,null,2,"""2""","""Appartement""",111.0,4,null,null,null,null,null,2.347559,48.843111,"""000AZ""",false,null,null,false,null,null,false,false,false
"""2024-1213647""",2024-04-22,1,"""Vente""",1.5e6,"""29""",null,"""RUE

### 5. Sélection des colonnes

In [86]:
col_to_keep = [
    "id_mutation",
    "date_mutation",
    "numero_disposition",
    "nature_mutation",
    "valeur_fonciere",
    "adresse_numero",
    "adresse_nom_voie",
    "code_postal",
    "code_commune",
    "nom_commune",
    "code_departement",
    "nombre_lots",
    "type_local",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "address_inferred_from_coords",
    "is_valeur_fonciere_missing",
    "is_surface_missing",
    "is_nombre_pieces_missing"
]

In [87]:
dvf_silver = dvf_silver[col_to_keep]
dvf_silver.sample(5)

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,address_inferred_from_coords,is_valeur_fonciere_missing,is_surface_missing,is_nombre_pieces_missing
str,date,i64,str,f64,str,str,str,str,str,str,i64,str,f64,i64,bool,bool,bool,bool
"""2021-1693596""",2021-09-17,1,"""Vente""",379700.0,"""23""","""RUE JEAN LECLAIRE""","""75017""","""75117""","""Paris 17e Arrondissement""","""75""",2,"""Dépendance""",null,0,false,false,true,false
"""2020-833192""",2020-11-19,1,"""Vente""",210000.0,"""5""","""RUE DES MONTIBOEUFS""","""75020""","""75120""","""Paris 20e Arrondissement""","""75""",2,"""Appartement""",23.0,1,false,false,false,false
"""2025-501500""",2025-04-15,1,"""Vente""",1.65e8,"""67""","""QUAI JACQUES CHIRAC""","""75007""","""75107""","""Paris 7e Arrondissement""","""75""",0,"""Dépendance""",null,0,false,false,true,false
"""2020-820692""",2020-10-16,1,"""Vente""",581400.0,"""10""","""RUE DES TAILLANDIERS""","""75011""","""75111""","""Paris 11e Arrondissement""","""75""",1,"""Appartement""",46.0,2,false,false,false,false
"""2024-1201685""",2024-09-06,1,"""Vente""",425000.0,"""15""","""RUE LORD BYRON""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",2,"""Appartement""",61.0,2,false,false,false,false


In [99]:
dvf_silver.null_count()

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,address_inferred_from_coords,is_valeur_fonciere_missing,is_surface_missing,is_nombre_pieces_missing
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,724,0,0,0,0,0,0,0,0,83312,1430,0,0,0,0


### 6. Lignes dupliquées

In [88]:
dvf_silver = dvf_silver.unique(maintain_order=True)

In [89]:
dvf_silver.shape

(205847, 19)

### 7. Valeurs invalides

In [90]:
dvf_silver.select(pl.col(pl.Int64, pl.Float64)).describe()

statistic,numero_disposition,valeur_fonciere,nombre_lots,surface_reelle_bati,nombre_pieces_principales
str,f64,f64,f64,f64,f64
"""count""",205847.0,205123.0,205847.0,122535.0,204417.0
"""null_count""",0.0,724.0,0.0,83312.0,1430.0
"""mean""",1.009094,1.8141e6,1.540348,69.507243,1.275633
"""std""",0.106505,1.0618e7,1.324535,252.427659,1.550966
"""min""",1.0,0.15,0.0,1.0,0.0
"""25%""",1.0,242857.0,1.0,27.0,0.0
"""50%""",1.0,455760.0,1.0,44.0,1.0
"""75%""",1.0,865000.0,2.0,71.0,2.0
"""max""",9.0,1.0034e9,236.0,28109.0,32.0


In [103]:
dvf_silver = dvf_silver.with_columns([
    pl.when((~pl.col("is_surface_missing")) 
            & (pl.col("surface_reelle_bati") <= 5)
    )
    .then(True)
    .otherwise(False)
    .alias("is_surface_tres_faible"),

    pl.when((~pl.col("is_valeur_fonciere_missing"))
            & (pl.col("valeur_fonciere") < 10_000)
    )
    .then(True)
    .otherwise(False)
    .alias("is_valeur_fonciere_tres_basse"),

    (
        (pl.col("type_local") == "Appartement") &
        (pl.col("nombre_lots") == 0)
    ).alias("is_appartement_sans_lot")])

In [107]:
dvf_silver = dvf_silver.with_columns([(
        (pl.col("is_surface_tres_faible")) |
        (pl.col("is_valeur_fonciere_tres_basse")) |
        (
            (pl.col("type_local") == "Appartement") &
            (pl.col("nombre_lots") == 0)
        )
    ).alias("is_transaction_atypique_agent")
])

In [108]:
dvf_silver.filter(pl.col("is_transaction_atypique_agent"))

id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,address_inferred_from_coords,is_valeur_fonciere_missing,is_surface_missing,is_nombre_pieces_missing,is_surface_tres_faible,is_valeur_fonciere_tres_basse,is_appartement_sans_lot,is_transaction_atypique_agent
str,date,i64,str,f64,str,str,str,str,str,str,i64,str,f64,i64,bool,bool,bool,bool,bool,bool,bool,bool
"""2020-814519""",2020-07-02,1,"""Vente""",1000.0,"""44""","""RUE DE MONTMORENCY""","""75003""","""75103""","""Paris 3e Arrondissement""","""75""",1,"""Appartement""",60.0,4,false,false,false,false,false,true,false,true
"""2020-814531""",2020-07-02,1,"""Vente""",4.0,"""51""","""AV MONTAIGNE""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",3,"""Appartement""",65.0,4,false,false,false,false,false,true,false,true
"""2020-814531""",2020-07-02,1,"""Vente""",4.0,"""51""","""AV MONTAIGNE""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",2,"""Appartement""",27.0,3,false,false,false,false,false,true,false,true
"""2020-814531""",2020-07-02,1,"""Vente""",4.0,"""51""","""AV MONTAIGNE""","""75008""","""75108""","""Paris 8e Arrondissement""","""75""",4,"""Appartement""",20.0,2,false,false,false,false,false,true,false,true
"""2020-814533""",2020-07-08,1,"""Vente""",2000.0,"""19""","""RUE DU ROULE""","""75001""","""75101""","""Paris 1er Arrondissement""","""75""",2,"""Appartement""",55.0,1,false,false,false,false,false,true,false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2021-1708383""",2021-09-09,1,"""Vente""",1.1515e6,"""7""","""RUE GUYTON DE MORVEAU""","""75013""","""75113""","""Paris 13e Arrondissement""","""75""",0,"""Appartement""",24.0,2,false,false,false,false,false,false,true,true
"""2021-1708383""",2021-09-09,1,"""Vente""",1.1515e6,"""7""","""RUE GUYTON DE MORVEAU""","""75013""","""75113""","""Paris 13e Arrondissement""","""75""",0,"""Appartement""",23.0,2,false,false,false,false,false,false,true,true
"""2021-1708429""",2021-09-17,1,"""Vente""",3500.0,"""10""","""RUE POPINCOURT""","""75011""","""75111""","""Paris 11e Arrondissement""","""75""",1,"""Dépendance""",null,0,false,false,true,false,false,true,false,true


***


# BPE bronze cleaning


### 1. Import

In [21]:
bpe_df = pl.read_csv(BRONZE_DIR / "DS_BPE_2024_data.csv",
                     separator=";",
                     infer_schema_length=0
                     )

In [4]:
bpe_df.shape

(2300480, 8)

In [5]:
bpe_df.head()

GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE
str,str,str,str,str,str,str,str
"""67""","""DEP""","""F""","""F1""","""F105""","""FACILITIES""","""2024""","""3"""
"""86""","""DEP""","""D""","""D6""","""D606""","""FACILITIES""","""2024""","""3"""
"""26""","""DEP""","""D""","""D6""","""D607""","""FACILITIES""","""2024""","""4"""
"""73""","""DEP""","""F""","""F1""","""F107""","""FACILITIES""","""2024""","""33"""
"""90""","""DEP""","""F""","""F1""","""F101""","""FACILITIES""","""2024""","""5"""


### 2. Typage des colonnes

In [22]:
DATE_COLS = ["TIME_PERIOD"]

In [23]:
INT_COLS = ["OBS_VALUE"]

In [6]:
STR_COLS = [
    "GEO",
    "GEO_OBJECT",
    "FACILITY_DOM",
    "FACILITY_SDOM",
    "FACILITY_TYPE",
    "BPE_MEASURE"
]

In [24]:
bpe_silver = bpe_df.with_columns(
    pl.col(DATE_COLS).str.to_date("%Y", strict=True).dt.year().cast(pl.Int64).alias("TIME_PERIOD"),
    pl.col(INT_COLS).cast(pl.Int64, strict=True)
)

In [24]:
bpe_silver.sample(5)

GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE
str,str,str,str,str,str,i64,i64
"""58033""","""COM""","""_T""","""_T""","""_T""","""FACILITIES""",2024,7
"""49328""","""COM""","""B""","""B3""","""B326""","""FACILITIES""",2024,15
"""42285""","""COM""","""F""","""F1""","""F114""","""FACILITIES""",2024,1
"""34032""","""BV2022""","""B""","""B2""","""B204""","""FACILITIES""",2024,55
"""31451""","""COM""","""D""","""D2""","""D274""","""FACILITIES""",2024,1


### 3. Slicing des données

In [25]:
code_arr = ["75101",
            "75102",
            "75103",
            "75104",
            "75105",
            "75106",
            "75107",
            "75108",
            "75109",
            "75110",
            "75111",
            "75112",
            "75113",
            "75114",
            "75115",
            "75116",
            "75117",
            "75118",
            "75119",
            "75120"]

In [26]:
bpe_silver = bpe_silver.filter(pl.col("GEO").is_in(code_arr))
bpe_silver.sample(5)

GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE
str,str,str,str,str,str,i64,i64
"""75107""","""ARM""","""B""","""_T""","""_T""","""FACILITIES""",2024,1043
"""75111""","""ARM""","""F""","""F1""","""F101""","""FACILITIES""",2024,5
"""75104""","""ARM""","""A""","""A2""","""A203""","""FACILITIES""",2024,21
"""75113""","""ARM""","""D""","""D2""","""D262""","""FACILITIES""",2024,8
"""75105""","""ARM""","""D""","""_T""","""_T""","""FACILITIES""",2024,1050


In [13]:
bpe_silver.shape

(3773, 8)

### 4. Valeurs manquantes

In [12]:
bpe_silver.null_count()

GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


Aucune valeur nulle identifiée

### 5. Lignes dupliquées

In [14]:
bpe_silver.unique().shape

(3773, 8)

Aucun doublon identifié

### 6. Ajustement des colonnes

In [27]:
df_codes_dom = pl.read_csv(BRONZE_DIR / "codes_FACILITY_DOM_BPE.csv",
                           separator=";",
                           infer_schema_length=0,
                           encoding="windows-1252"
                           )
df_codes_dom = df_codes_dom.rename({"libelle français":"classification_n1"})
df_codes_dom

code,classification_n1
str,str
"""_T""","""Total"""
"""A""","""Services pour les particuliers"""
"""B""","""Commerces"""
"""C""","""Enseignement"""
"""D""","""Santé et action sociale"""
"""E""","""Transports et déplacements"""
"""F""","""Sports, loisirs et culture"""
"""G""","""Tourisme"""


In [28]:
df_codes_sdom = pl.read_csv(BRONZE_DIR / "codes_FACILITY_SDOM_BPE.csv",
                            separator=";",
                            infer_schema_length=0,
                            encoding="windows-1252"
                            )
df_codes_sdom = df_codes_sdom.rename({"libelle français":"classification_n2"})
df_codes_sdom.sample(5)

code,classification_n2
str,str
"""D1""","""Etablissements et services de …"
"""A3""","""Services automobiles"""
"""C2""","""Enseignement du second degré -…"
"""A1""","""Services publics"""
"""D6""","""Action sociale pour handicapés"""


In [29]:
df_codes_type = pl.read_csv(BRONZE_DIR / "codes_FACILITY_TYPE_BPE.csv",
                            separator=";",
                            infer_schema_length=0,
                            encoding="windows-1252"
                            )
df_codes_type = df_codes_type.rename({"libelle français":"classification_n3"})
df_codes_type.sample(5)

code,classification_n3
str,str
"""D270""","""Spécialiste en ophtalmologie"""
"""F110""","""Sports de glace"""
"""B311""","""Horlogerie-bijouterie"""
"""D115""","""Services de santé maternelle e…"
"""C410""","""école de formation aux profess…"


In [30]:
bpe_silver = bpe_silver.join(df_codes_dom,
                        left_on=["FACILITY_DOM"],
                        right_on=["code"],
                        how="left",
                        validate="m:1"
                        )

In [31]:
bpe_silver = bpe_silver.join(df_codes_sdom,
                        left_on=["FACILITY_SDOM"],
                        right_on=["code"],
                        how="left",
                        validate="m:1"
                        )

In [32]:
bpe_silver = bpe_silver.join(df_codes_type,
                        left_on=["FACILITY_TYPE"],
                        right_on=["code"],
                        how="left",
                        validate="m:1"
                        )

In [33]:
bpe_silver = bpe_silver.with_columns((pl.lit("750") + pl.col("GEO")
                                     .str.slice(-2))
                                     .alias("code_postal")
                                     )
bpe_silver.sample(5)

GEO,GEO_OBJECT,FACILITY_DOM,FACILITY_SDOM,FACILITY_TYPE,BPE_MEASURE,TIME_PERIOD,OBS_VALUE,classification_n1,classification_n2,classification_n3,code_postal
str,str,str,str,str,str,i64,i64,str,str,str,str
"""75108""","""ARM""","""A""","""A5""","""A502""","""FACILITIES""",2024,5,"""Services pour les particuliers""","""Autres services""","""Vétérinaire""","""75008"""
"""75105""","""ARM""","""B""","""B3""","""B318""","""FACILITIES""",2024,15,"""Commerces""","""Commerces spécialisés non-alim…","""Commerce de jeux et jouets""","""75005"""
"""75101""","""ARM""","""D""","""D2""","""D250""","""FACILITIES""",2024,65,"""Santé et action sociale""","""Fonctions médicales et paraméd…","""Psychologue""","""75001"""
"""75102""","""ARM""","""D""","""D2""","""D265""","""FACILITIES""",2024,24,"""Santé et action sociale""","""Fonctions médicales et paraméd…","""Médecin généraliste""","""75002"""
"""75115""","""ARM""","""A""","""A3""","""_T""","""FACILITIES""",2024,139,"""Services pour les particuliers""","""Services automobiles""","""Total""","""75015"""


In [34]:
bpe_silver = bpe_silver.rename({"TIME_PERIOD":"annee", "OBS_VALUE":"nombre_equipements"})

In [60]:
test = bpe_silver["BPE_MEASURE"].unique().to_list()
test

['FACILITIES']

In [35]:
col_to_keep = ["annee",
               "code_postal",
               "classification_n1",
               "classification_n2",
               "classification_n3",
               "nombre_equipements"
               ]

In [36]:
bpe_silver = bpe_silver[col_to_keep]

In [63]:
bpe_silver.sample(5)

annee,code_postal,classification_n1,classification_n2,classification_n3,nombre_equipements
i64,str,str,str,str,i64
2024,"""75008""","""Tourisme""","""Tourisme""","""Hôtel""",153
2024,"""75012""","""Santé et action sociale""","""Fonctions médicales et paraméd…","""Médecin généraliste""",125
2024,"""75014""","""Services pour les particuliers""","""Artisanat du bâtiment""","""Plâtrier peintre""",165
2024,"""75013""","""Services pour les particuliers""","""Services publics""","""Total""",9
2024,"""75010""","""Santé et action sociale""","""Action sociale pour enfants en…","""Relais petite enfance""",2


In [37]:
bpe_silver.shape

(3773, 6)

In [38]:
bpe_silver.head()

annee,code_postal,classification_n1,classification_n2,classification_n3,nombre_equipements
i64,str,str,str,str,i64
2024,"""75004""","""Santé et action sociale""","""Fonctions médicales et paraméd…","""Spécialiste en cardiologie""",26
2024,"""75004""","""Commerces""","""Commerces spécialisés non-alim…","""Station de recharge de véhicul…",13
2024,"""75003""","""Santé et action sociale""","""Fonctions médicales et paraméd…","""Spécialiste en pédiatrie""",4
2024,"""75014""","""Santé et action sociale""","""Fonctions médicales et paraméd…","""Spécialiste en oncologie, anat…",38
2024,"""75010""","""Commerces""","""Commerces alimentaires""","""Supérette""",26


In [6]:
test = bpe_df.filter(pl.Series(bpe_df.select("GEO")).str.contains("^751.*"))
res = test.select("GEO").to_series().unique().to_list()
len(res)

31

In [ ]:
unique_geo = pl.Series(bpe_df.select("GEO"))
unique_geo.str.contains("751")